# Test out counterfactual simulation probabilities

## 1. Load the datasets and explainers

In [ ]:
import os

# # Store the current directory
# current_directory = os.getcwd()
# # Change to the parent directory only if not already in the parent
# if os.path.basename(current_directory) != os.path.basename(os.path.abspath(os.path.join(current_directory, os.pardir))):
#     os.chdir(os.pardir)

from src.utils import AIDatasetLoader, filter_by_app_and_model, DecisionTreeInterpreter, LogisticRegressionInterpreter  
from src.memory import Chunk, DeclarativeMemory

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time


In [ ]:
current_dir = os.getcwd()
data_dir = os.path.join(current_dir, 'datasets')

file_values = os.path.join(data_dir, 'values.csv')
file_metadata = os.path.join(data_dir, 'metadata.csv')
file_prediction = os.path.join(data_dir, 'none.csv')

values_df = pd.read_csv(file_values)
metadata_df = pd.read_csv(file_metadata)
prediction_df = pd.read_csv(file_prediction)

# ✅ Which model to use per dataset
dataset_model_map = {
    "mushrooms": "mlp",
    "wine_quality": "mlp",
    "forest_cover": "xgboost",
    "adult": "xgboost",
}

# # 🔧 Choose dataset here:
# app_id = "wine_quality"  # 🔄 Change this line to switch datasets

# # 🧠 Auto-configured values:
# model_name = dataset_model_map[app_id]

# load ai loader
ai_dataset_loader = AIDatasetLoader(
    feature_values_df=values_df,
    metadata_df=metadata_df,
    AI_predictions_df=prediction_df
)

# Load decision tree
dt_df = pd.read_csv(os.path.join(data_dir, 'decision_tree.csv'))
# dt_exp = DecisionTreeInterpreter(dt_df, metadata_df, app_id, model_name, depth=2)
# dt_exp.print_tree(as_name=True)


# Load linear model
lr_df = pd.read_csv(os.path.join(data_dir, 'logistic_regression.csv'))
# lr_exp = LogisticRegressionInterpreter(lr_df, metadata_df, app_id, model_name, variant="sparse")


In [ ]:
import models
from tabular_datasets.tabular_dataset import TabularDataset

def load_data(file_path):
    base_path = os.path.dirname(file_path)
    metadata = pd.read_csv(os.path.join(base_path, 'metadata.csv')).iloc[0].to_dict()

    for key in ['feature_names', 'target_options', 'ordinal_feature_indices',
                'categorical_feature_options', 'feature_boundaries']:
        metadata[key] = eval(metadata[key])

    X = np.load(os.path.join(base_path, 'X.npy'))
    y = np.load(os.path.join(base_path, 'y.npy'))

    return TabularDataset(X, y,
                          feature_names=metadata['feature_names'],
                          categorical_feature_options=metadata['categorical_feature_options'],
                          ordinal_feature_indices=metadata['ordinal_feature_indices'],
                          target_name=metadata['target_name'],
                          target_options=metadata['target_options'],
                          feature_boundaries=metadata['feature_boundaries'],
                          dataset_name=metadata['dataset_name'])

def load_transform_and_ai(app_id):
    model_path = os.path.join(os.getcwd(), "models/MLP", f"{app_id}_model_weights.pth")

    path = os.path.join("tabular_datasets1", f"{app_id}/train/")
    train_data = load_data(path)

    X_train, y_train = train_data.prepare_data_for_model(one_hot_encode=True)
    input_dim = X_train.shape[1]
    num_classes = len(train_data.target_options)

    ai = models.get_model("mlp", input_dim=input_dim, num_classes=num_classes)
    ai.load(model_path)

    # ai.predict(train_data.prepare_instances_for_model(train_data.X[0], one_hot_encode=True))
    return train_data, ai



def run_ai_prediction(instance, transform, ai):
    instance = np.array(instance)
    prepared_instance = transform.prepare_instances_for_model(instance, one_hot_encode=True)
    preds = ai.predict(prepared_instance)
    return np.argmax(preds[0])


In [ ]:
# ✅ Which model to use per dataset
dataset_model_map = {
    "mushrooms": "mlp",
    "wine_quality": "mlp",
    "forest_cover": "mlp",
    "adult": "mlp",
}
# 🔧 Choose datasets (only wine_quality and mushrooms)
dataset_mapping = {
    1: "wine_quality",
    2: "mushrooms",
}

# Auto-configure model names
app_id1 = dataset_mapping[1]
app_id2 = dataset_mapping[2]
model_name1 = dataset_model_map[app_id1]
model_name2 = dataset_model_map[app_id2]

# --- Load dataset subsets ---
ai_dataset_loader = AIDatasetLoader(
    feature_values_df=values_df,
    metadata_df=metadata_df,
    AI_predictions_df=prediction_df,
)

ai_data_loader1 = filter_by_app_and_model(ai_dataset_loader, app_id1, model_name1)
ai_data_loader2 = filter_by_app_and_model(ai_dataset_loader, app_id2, model_name2)

# --- Decision Tree explainers ---
dt_df = pd.read_csv(os.path.join(data_dir, "decision_tree.csv"))
dt_exp1 = DecisionTreeInterpreter(dt_df, metadata_df, app_id1, model_name1)
dt_exp2 = DecisionTreeInterpreter(dt_df, metadata_df, app_id2, model_name2)
dt_exp3 = DecisionTreeInterpreter(dt_df, metadata_df, app_id1, model_name1, depth=2)
dt_exp4 = DecisionTreeInterpreter(dt_df, metadata_df, app_id2, model_name2, depth=2)
dt_exp5 = DecisionTreeInterpreter(dt_df, metadata_df, app_id1, model_name1, depth=3)
dt_exp6 = DecisionTreeInterpreter(dt_df, metadata_df, app_id2, model_name2, depth=3)

# --- Logistic Regression explainers ---
lr_df = pd.read_csv(os.path.join(data_dir, "logistic_regression.csv"))
lr_exp1 = LogisticRegressionInterpreter(lr_df, metadata_df, app_id1, model_name1)
lr_exp2 = LogisticRegressionInterpreter(lr_df, metadata_df, app_id2, model_name2)
lr_exp3 = LogisticRegressionInterpreter(lr_df, metadata_df, app_id1, model_name1, variant="sparse")
lr_exp4 = LogisticRegressionInterpreter(lr_df, metadata_df, app_id2, model_name2, variant="sparse")

# --- Dictionaries for environment ---
ai_dataset_loaders = {
    1: ai_data_loader1,
    2: ai_data_loader2,
    3: ai_data_loader1,
    4: ai_data_loader2,
}

dt_exps = {
    1: dt_exp1,
    2: dt_exp2,
    3: dt_exp3,
    4: dt_exp4,
}

lr_exps = {
    1: lr_exp1,
    2: lr_exp2,
    3: lr_exp3,
    4: lr_exp4,
}

# --- Load AIs & transforms for 1–4 ---
ais = {}
transforms = {}
for i in range(1, 5):
    dataset_name = dataset_mapping[((i-1) % 2) + 1]  # cycles 1→2→1→2...
    train_data, ai = load_transform_and_ai(dataset_name)
    ais[i] = ai
    transforms[i] = train_data


In [ ]:
# import importlib
# import src.memory as memory
# importlib.reload(memory)
# import src.dt_memory as dt_memory
# importlib.reload(dt_memory)
# import src.heuristic_lr_model as heuristic_lr_model
# importlib.reload(heuristic_lr_model)
# import src.lr_memory as lr_memory
# importlib.reload(lr_memory)

# from src.memory import DeclarativeMemory, CombinedMemory
# from src.dt_memory import (
#     add_dt_to_memory, dt_traverse, refresh_dt_path_in_memory
# )
# from src.heuristic_lr_model import (
#     add_lr_heuristic_to_memory, lr_heuristic, refresh_lr_heuristic_in_memory
# )
# from src.lr_memory import (
#     add_lr_calculation_to_memory, lr_calculation, refresh_lr_calculation_in_memory
# )

# from typing import Optional
# import random
# from dataclasses import replace


In [ ]:

import importlib
import src.memory as memory
importlib.reload(memory)
import src.dt_memory as dt_memory
importlib.reload(dt_memory)
import src.heuristic_lr_model as heuristic_lr_model
importlib.reload(heuristic_lr_model)
import src.lr_memory as lr_memory
importlib.reload(lr_memory)

from src.memory import DeclarativeMemory, CombinedMemory
from src.dt_memory import (
    add_dt_to_memory, dt_traverse, refresh_dt_path_in_memory, cf_change_path_dt, recall_change_dt
)
from src.heuristic_lr_model import (
    add_lr_heuristic_to_memory, lr_heuristic, refresh_lr_heuristic_in_memory, cf_lr_heuristic
)
from src.lr_memory import (
    add_lr_calculation_to_memory, lr_calculation, refresh_lr_calculation_in_memory, cf_lr_calculation, recall_change_lr
)

import math
import os
import numpy as np
import pandas as pd
from typing import Iterable, Union, Optional
import random
from dataclasses import dataclass, replace 


import pandas as pd
import numpy as np
from typing import Dict, List, Optional, Union


In [ ]:
# ---------------- Memory builder ----------------
def _make_memory(memory_recall_threshold):
    dm = DeclarativeMemory(
        memory_recall_threshold=memory_recall_threshold,
        cue_association_strength=2.0,
        memory_mismatch_penalty=-2.0,
        memory_recall_noise=0.3,
    )
    return CombinedMemory(dm, working_memory_capacity=7)


## Preparation functions and final calls

In [ ]:
def prepare_memory_for_dt(memory, dt_exp, ai_loader, forward_trials, bounds):
    add_dt_to_memory(memory, dt_exp)

    memory.tick(90)

    for trial in forward_trials:
        with_xai = trial['Tested w/ XAI']
        instance_id = trial['Instance Id']
        instances, preds = ai_loader.load_instances([instance_id], normalize=False)
        instance = instances[0]

        explanation_access_mode = "read" if with_xai==1 else "retrieve"

        dt_traverse(instance, memory, dt_exp, explanation_access_mode=explanation_access_mode, read_seconds_per_item=1.0, decision_boundary=1.0, decision_noise=0.8)
        if explanation_access_mode=="read":
            refresh_dt_path_in_memory(memory, dt_exp, instance)

def prepare_memory_for_lr_heuristic(memory, lr_exp, ai_loader, forward_trials, bounds):
    add_lr_heuristic_to_memory(lr_exp, memory)

    memory.tick(90)

    for trial in forward_trials:
        instance_id = trial['Instance Id']
        instances, preds = ai_loader.load_instances([instance_id], normalize=True)
        instance = instances[0]

        p, t, info = lr_heuristic(instance, memory, lr_exp, read_seconds_per_item=1.0, decision_boundary=1.0, decision_noise=0.8)
        refresh_lr_heuristic_in_memory(memory, lr_exp, info, actual=trial['AI prediction'])


In [ ]:
def sample_from_probs(probs):
    features = [k for k in probs.keys() if k != 'expected_time']
    distribution = [probs[k]['p_selected'] for k in features]
    if sum(distribution)!=1:
        distribution = [p/sum(distribution) for p in distribution]
    sampled_feature = np.random.choice(features, p=distribution)
    return sampled_feature, probs[sampled_feature]['mean_delta']

def apply_change_to_feature(instance, feature_name, bounds, delta, counterfactual_overshoot_fraction=0.1):
    instance = instance.copy()
    index = int(feature_name[-1])

    original_value = instance[index]

    counterfactual_overshoot_fraction = (bounds[feature_name][1] - bounds[feature_name][0]) * counterfactual_overshoot_fraction
    counterfactual_overshoot_fraction = -counterfactual_overshoot_fraction if delta < 0 else counterfactual_overshoot_fraction

    new_value = original_value + delta + counterfactual_overshoot_fraction
    new_value = max(bounds[feature_name][0], min(bounds[feature_name][1], new_value))

    instance[index] = new_value
    return instance

def smooth_probs_with_random_response(probs, random_response_rate, num_features=6):
    smoothed_probs = probs.copy()
    # Get 'a{i}' from 0-5
    features = [f'a{i}' for i in range(num_features)]

    for feature in features:
        feature_info = probs.get(feature, {})
        p = feature_info.get('p_selected', 0.0)
        new_p = (1 - random_response_rate) * p + (random_response_rate / num_features)
        if feature_info!={}:
            smoothed_probs[feature]['p_selected'] = new_p
        else:
            smoothed_probs[feature] = {}
            smoothed_probs[feature]['p_selected'] = new_p
            smoothed_probs[feature]['mean_delta'] = 0.0
        # if feature == 'expected_time':
        #     smoothed_probs[feature] = values
        # else:
        #     p_selected = values['p_selected']
        #     try:
        #         smoothed_p = (1 - random_response_rate) * p_selected + (random_response_rate / num_features)
        #     except:
        #     smoothed_probs[feature] = {
        #         'p_selected': smoothed_p,
        #         'mean_delta': values['mean_delta']
        #     }
    return smoothed_probs


In [ ]:
def zero_out_lr_heuristic(norm_instance, memory, lr_exp, bounds, actual_label, random_response_rate=0.1):
    out = cf_lr_heuristic(norm_instance, memory, lr_exp, bounds, actual_label=actual_label, retrieval_candidate_count=6)
    time = out.pop('expected_time')

    smoothed_out = smooth_probs_with_random_response(out, random_response_rate=random_response_rate)

    memory.tick(time)    

    feat, delta = sample_from_probs(smoothed_out)

    return feat, delta, time

def zero_out_lr_displayed(instance, memory, lr_exp, bounds, actual_label, random_response_rate=0.1):
    out = cf_lr_calculation(instance, lr_exp, bounds=bounds, memory=memory)
    time = out.pop('expected_time')
    memory.tick(time)

    smoothed_out = smooth_probs_with_random_response(out, random_response_rate=random_response_rate)

    feat, delta = sample_from_probs(smoothed_out)

    xai_pred = int(lr_exp.apply_to_instance(instance)>0)

    if xai_pred != actual_label:
        delta = -delta

    return feat, delta, time

def change_dt_path(instance, memory, dt_exp, bounds, actual_label, depth=1, explanation_access_mode="retrieve", random_response_rate=0.1):
    out = cf_change_path_dt(instance, dt_exp, bounds, memory=memory, counterfactual_tree_depth=depth, explanation_access_mode=explanation_access_mode, depth_choice_temperature=0.2)
    time = out.pop('expected_time')
    memory.tick(time)

    smoothed_out = smooth_probs_with_random_response(out, random_response_rate=random_response_rate)

    feat, delta = sample_from_probs(smoothed_out)

    xai_pred = dt_exp.apply_to_instance(instance)['class_index']

    if xai_pred != actual_label:
        delta = -delta

    return feat, delta, time

def recall_change_dt_full(instance, memory, bounds, random_response_rate=0.1):
    out = recall_change_dt(instance, memory, bounds=bounds, retrieved_combo_count=3)
    time = out.pop('expected_time')
    memory.tick(time)

    smoothed_out = smooth_probs_with_random_response(out, random_response_rate=random_response_rate)

    feat, delta = sample_from_probs(smoothed_out)

    return feat, delta, time

def recall_change_lr_full(instance, memory, bounds, actual_label, random_response_rate=0.1):
    
    direction = 'increase' if actual_label==0 else 'decrease'

    out = recall_change_lr(memory, retrieved_combo_count=6, preferred_change_direction=direction)
    time = out.pop('expected_time')
    memory.tick(time)

    smoothed_out = smooth_probs_with_random_response(out, random_response_rate=random_response_rate)

    feat, delta = sample_from_probs(smoothed_out)

    return feat, delta, time


## RL environment setup

In [ ]:
strategies = {
    0: "change_path_dt",
    1: "zero_out_lr_heuristic",
    2: "zero_out_lr_displayed",
    3: "recall_change_dt",
    4: "recall_change_lr"
}

XAI_types = {
    0: "DT",
    1: "LR",
    2: "DT+LR"
}


In [ ]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces


In [ ]:
class CounterfactualEnv(gym.Env):    
    def __init__(self,
        ai_dataset_loaders,
        ais,
        transforms,
        lr_exps,
        dt_exps,
        cog_params,
        instances_per_episode = 40,
        max_features = 6,
        eval_overrides = None,
        ):
        super().__init__()


        self.ai_dataset_loaders = ai_dataset_loaders
        self.ais = ais
        self.transforms = transforms
        self.lr_exps = lr_exps
        self.dt_exps = dt_exps
        self.cog_params = cog_params
        self.instances_per_episode = instances_per_episode
        self.max_features = max_features

        self.eval_overrides = eval_overrides or {}

        
        self.chi_low, self.chi_high = cog_params.get('time_penalty_weight')

        self.action_space = spaces.MultiDiscrete([5, 3])  # 5 strategies, depth for the decision tree choice

        # Varied cognitive params
        self.varied_cogparams_low = []
        self.varied_cogparams_high = []
        self.num_varied_cogparam = 0
        self.varied_param_names = []
        for k, v in self.cog_params.items():
            # we will treat any (lo,hi) param as "varied" *except* time_penalty_weight which is handled above
            if k in ["time_penalty_weight"]:
                continue
            if isinstance(v, (list, tuple)) and len(v) == 2:
                self.num_varied_cogparam += 1
                self.varied_param_names.append(k)
                self.varied_cogparams_low.append(float(v[0]))
                self.varied_cogparams_high.append(float(v[1]))

        low_vec = [self.chi_low, 0.0, 0.0, 0.0, 0.0] + [0.0] * 3 * len(strategies) # time_penalty_weight value, trial_index, with_xai, condition, xai_shown + (counts, success rate, mean time) for each strategy
        high_vec = [self.chi_high, float(self.instances_per_episode-1), 1.0, float(len(XAI_types.keys()) - 1), 1] + [self.instances_per_episode, 1.0, 30.0] * len(strategies) # time_penalty_weight value, trial_index, with_xai + (counts, success rate, mean time) for each strategy

        low_vec += self.varied_cogparams_low
        high_vec += self.varied_cogparams_high

        self.observation_space = spaces.Box(
            low=np.array(low_vec, dtype=np.float32),
            high=np.array(high_vec, dtype=np.float32),
            dtype=np.float32
        )

        self.step_idx = 0.0
        self.curr_chi = 0.0

        self.dt_memory = None
        self.lr_memory = None
        self.lr_exp = None
        self.dt_exp = None
        self.ai_dataset_loader = None
        self.bounds = None
        self.ai = None

        self.xai_type = None
        self.xai_schedule = None

        self.current_cog_params = {}
        self.with_xai_schedule = None

        self.counts = None
        self.success_rates = None
        self.mean_times = None

        self.forward_trials = None
        self.counterfactual_trials = None

    def _initialize_memories(self):
        self.dt_memory = _make_memory(
            memory_recall_threshold=self.current_cog_params.get('memory_recall_threshold', -1.0),
        )
        self.lr_memory = _make_memory(
            memory_recall_threshold=self.current_cog_params.get('memory_recall_threshold', -1.0),
        )

        prepare_memory_for_dt(self.dt_memory, self.dt_exp, self.ai_dataset_loader, self.forward_trials, self.bounds)
        prepare_memory_for_lr_heuristic(self.lr_memory, self.lr_exp, self.ai_dataset_loader, self.forward_trials, self.bounds)


    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        rng = np.random.default_rng(seed)

        # --- choose dataset/ais as before ---
        self.key = rng.choice(list(self.ai_dataset_loaders.keys()))
        self.ai_dataset_loader = self.ai_dataset_loaders[self.key]
        self.ai = self.ais[self.key]
        self.transform = self.transforms[self.key]
        self.lr_exp = self.lr_exps[self.key]
        self.dt_exp = self.dt_exps[self.key]
        self.app_id = self.lr_exp.app_id
        self.bounds = self.ai_dataset_loader.get_bounds_for_app(app_id=self.app_id)

        # ===== (A) Force XAI type/schedule when provided =====
        force_xai_type = self.eval_overrides.get("xai_type", None)  # 'DT' | 'LR' | 'DT+LR' | None
        if force_xai_type is not None:
            self.xai_type = force_xai_type
        else:
            self.xai_type = rng.choice(list(XAI_types.keys()))

        # self.xai_type is the condition
        if XAI_types[self.xai_type] == 'DT':
            self.xai_schedule = ['DT'] * self.instances_per_episode
        elif XAI_types[self.xai_type] == 'LR':
            self.xai_schedule = ['LR'] * self.instances_per_episode
        else:  # 'DT+LR'
            self.xai_schedule = rng.choice(['DT','LR'], size=self.instances_per_episode, p=[0.5,0.5]).tolist()

        # allow full override of xai_schedule if provided (list[str] of len=instances_per_episode)
        if "xai_schedule" in self.eval_overrides:
            xs = self.eval_overrides["xai_schedule"]
            assert len(xs) == self.instances_per_episode
            self.xai_schedule = list(xs)

        # ===== (B) Force With-XAI ratio/schedule =====
        with_xai_prob = float(self.eval_overrides.get("with_xai_prob", 0.5))
        self.with_xai_schedule = rng.choice([0,1], size=self.instances_per_episode,
                                            p=[1.0-with_xai_prob, with_xai_prob])
        if "with_xai_schedule" in self.eval_overrides:
            ws = self.eval_overrides["with_xai_schedule"]
            assert len(ws) == self.instances_per_episode
            self.with_xai_schedule = np.asarray(ws, dtype=int)

        # --- pick forward/cf instances as before ---
        forward_ids = rng.choice(range(400), size=self.instances_per_episode, replace=False)
        instances, _ = self.ai_dataset_loader.load_instances(list(forward_ids), normalize=False)
        ai_preds = [run_ai_prediction(instance, self.transform, self.ai) for instance in instances]
        self.forward_trials = [
            {'Tested w/ XAI': int(self.with_xai_schedule[i]), 'Instance Id': int(forward_ids[i]), 'AI prediction': int(ai_preds[i])}
            for i in range(self.instances_per_episode)
        ]

        counterfactual_ids = rng.choice(range(400), size=self.instances_per_episode, replace=False)
        self.counterfactual_trials = [
            {'Tested w/ XAI': int(self.with_xai_schedule[i]), 'Instance Id': int(counterfactual_ids[i])}
            for i in range(self.instances_per_episode)
        ]

        # ===== (C) Freeze cognitive params if provided =====
        self.current_cog_params = {}
        force_cog = self.eval_overrides.get("cog_params_fixed", None)
        if force_cog is not None:
            # use provided values, but keep types float
            for k,v in force_cog.items():
                self.current_cog_params[k] = float(v)
            # if time_penalty_weight provided, also clamp curr_chi to it
            if "time_penalty_weight" in force_cog:
                self.curr_chi = float(force_cog["time_penalty_weight"])
        else:
            for k, v in (self.cog_params or {}).items():
                if isinstance(v, (list, tuple)) and len(v) == 2:
                    self.current_cog_params[k] = float(np.random.uniform(v[0], v[1]))
                elif isinstance(v, (int, float)):
                    self.current_cog_params[k] = float(v)

        # initialize memories after current_cog_params
        self._initialize_memories()

        self.step_idx = 0
        if force_cog is None or "time_penalty_weight" not in force_cog:
            self.curr_chi = np.random.uniform(self.chi_low, self.chi_high)

        self.counts = {k: 0 for k in strategies.keys()}
        self.success_rates = {k: 0.0 for k in strategies.keys()}
        self.mean_times = {k: 0.0 for k in strategies.keys()}

        self.xai_type_shown = self.xai_schedule[0]  # initialize to first trial's xai_type

        obs = self._build_obs()
        return obs, {}

    # def reset(self, seed = None, options = None):
    #     super().reset(seed=seed)
    #     rng = np.random.default_rng(seed)

    #     # choose dataset and explainers
    #     self.key = rng.choice(list(self.ai_dataset_loaders.keys()))
    #     self.ai_dataset_loader = self.ai_dataset_loaders[self.key]
    #     self.ai = self.ais[self.key]
    #     self.transform = self.transforms[self.key]
    #     self.lr_exp = self.lr_exps[self.key]
    #     self.dt_exp = self.dt_exps[self.key]

    #     self.app_id = self.lr_exp.app_id
    #     self.bounds = self.ai_dataset_loader.get_bounds_for_app(app_id=self.app_id)

    #     self.xai_type = rng.choice(list(XAI_types.keys()))
    #     if XAI_types[self.xai_type] == 'DT':
    #         self.xai_schedule = ['DT'] * self.instances_per_episode
    #     elif XAI_types[self.xai_type] == 'LR':
    #         self.xai_schedule = ['LR'] * self.instances_per_episode
    #     elif XAI_types[self.xai_type] == 'DT+LR':
    #         self.xai_schedule = rng.choice(['DT', 'LR'], size=self.instances_per_episode, p=[0.5, 0.5]).tolist()

    #     # With-XAI schedule
    #     self.with_xai_schedule = rng.choice([0,1], size=self.instances_per_episode, p=[0.5,0.5])
        
    #     forward_ids = rng.choice(range(400), size=self.instances_per_episode, replace=False)
    #     instances, _ = self.ai_dataset_loader.load_instances(list(forward_ids), normalize=False)
    #     ai_preds = [run_ai_prediction(instance, self.transform, self.ai) for instance in instances]
    #     # self.forward_trials = {"Tested w/ XAI": self.with_xai_schedule, "Instance Id": forward_ids.tolist(), "AI prediction": ai_preds}
    #     self.forward_trials = [({'Tested w/ XAI': self.with_xai_schedule[i], 'Instance Id': forward_ids[i], 'AI prediction': ai_preds[i]}) for i in range(self.instances_per_episode)]

    #     counterfactual_ids = rng.choice(range(400), size=self.instances_per_episode, replace=False)
    #     self.counterfactual_trials = [({'Tested w/ XAI': self.with_xai_schedule[i], 'Instance Id': counterfactual_ids[i]}) for i in range(self.instances_per_episode)]

    #     self.current_cog_params = {}
    #     for k, v in (self.cog_params or {}).items():
    #         if isinstance(v, (list, tuple)) and len(v) == 2:
    #             self.current_cog_params[k] = float(rng.uniform(v[0], v[1]))
    #         elif isinstance(v, (int, float)):
    #             self.current_cog_params[k] = float(v)

    #     self._initialize_memories()

    #     self.step_idx = 0
    #     self.curr_chi = np.random.uniform(self.chi_low, self.chi_high)
    #     self.counts = {k: 0 for k in strategies.keys()}
    #     self.success_rates = {k: 0.0 for k in strategies.keys()}
    #     self.mean_times = {k: 0.0 for k in strategies.keys()}

    #     obs = self._build_obs()
    #     return obs, {}

    def _build_obs(self):
        if self.step_idx >= self.instances_per_episode:
            return np.zeros(self.observation_space.shape, dtype=np.float32)

        xai_type_shown_index = [k for k in XAI_types.keys() if XAI_types[k] == self.xai_type_shown][0]

        obs = [self.curr_chi, float(self.step_idx), float(self.with_xai_schedule[int(self.step_idx)])]
        obs += [float(self.xai_type), float(xai_type_shown_index)]  # xai_type, xai_type_shown
        for k in strategies.keys():
            obs += [float(self.counts[k]), float(self.success_rates[k]), float(self.mean_times[k])]
        
        for name in self.varied_param_names:
            obs.append(float(self.current_cog_params[name]))

        return np.array(obs, dtype=np.float32)
    
    def step(self, action):
        strategy, depth = action
        strategy = int(strategy)
        depth = int(depth)

        trial = self.counterfactual_trials[int(self.step_idx)]
        instance_id = trial['Instance Id']
        with_xai = trial['Tested w/ XAI']

        instances, preds = self.ai_dataset_loader.load_instances([instance_id], normalize=False)
        instance = instances[0]
        norm_instance = self.ai_dataset_loader.load_instances([instance_id], normalize=True)[0][0]

        strategy_name = strategies[strategy]
        actual_label = preds[0]

        explanation_access_mode = "retrieve"
        invalid_strategy = False
        self.xai_type_shown = self.xai_schedule[int(self.step_idx)]
        if self.xai_type_shown == 'DT':
            explanation_access_mode = 'retrieve' if with_xai==0 else 'read'
            if strategy_name == 'zero_out_lr_displayed':
                invalid_strategy = True

        # strategy not allowed
        if XAI_types[self.xai_type]=='DT':
            if strategy_name in ['zero_out_lr_displayed', 'zero_out_lr_heuristic', 'recall_change_lr']:
                invalid_strategy = True
        elif XAI_types[self.xai_type]=='LR':
            if strategy_name in ['change_path_dt', 'recall_change_dt']:
                invalid_strategy = True
            if strategy_name == 'zero_out_lr_displayed' and with_xai==0:
                invalid_strategy = True
        
        if invalid_strategy:
            self.step_idx += 1
            done = self.step_idx >= self.instances_per_episode
            obs = self._build_obs()
            info = {
                "error": "invalid strategy",
                "strategy": strategy_name,
                "depth": depth if strategy_name=="change_path_dt" else None,
                "instance_id": instance_id,
                "with_xai": with_xai,
                "xai_type": self.xai_type,
                "xai_type_shown": self.xai_type_shown,
                "invalid_under_condition": True,
                "time": 0.0,
                "success": 0,
            }
            # small penalty for invalid choice
            return obs, -1.0, False, done, info

        if strategy_name == "change_path_dt":
            feat, delta, time = change_dt_path(instance, self.dt_memory, self.dt_exp, self.bounds, actual_label, depth=depth, explanation_access_mode=explanation_access_mode, random_response_rate=self.current_cog_params.get('random_response_rate', 0.1))
        elif strategy_name == "zero_out_lr_heuristic":
            feat, delta, time = zero_out_lr_heuristic(norm_instance, self.lr_memory, self.lr_exp, self.bounds, actual_label, random_response_rate=self.current_cog_params.get('random_response_rate', 0.1))
        elif strategy_name == "zero_out_lr_displayed":
            feat, delta, time = zero_out_lr_displayed(instance, self.lr_memory, self.lr_exp, self.bounds, actual_label, random_response_rate=self.current_cog_params.get('random_response_rate', 0.1))
        elif strategy_name == "recall_change_dt":
            feat, delta, time = recall_change_dt_full(instance, self.dt_memory, self.bounds, random_response_rate=self.current_cog_params.get('random_response_rate', 0.1))
        elif strategy_name == "recall_change_lr":
            feat, delta, time = recall_change_lr_full(instance, self.lr_memory, self.bounds, actual_label, random_response_rate=self.current_cog_params.get('random_response_rate', 0.1))

        new_instance = apply_change_to_feature(instance, feat, self.bounds, delta, counterfactual_overshoot_fraction=self.current_cog_params.get('counterfactual_overshoot_fraction', 0.1))

        new_pred = run_ai_prediction(new_instance, self.transform, self.ai)

        old_pred = run_ai_prediction(instance, self.transform, self.ai)


        # if strategy_name == "recall_change_dt":

        success = 1 if new_pred != actual_label else 0

        self.counts[strategy] += 1
        self.success_rates[strategy] = ((self.success_rates[strategy] * (self.counts[strategy]-1)) + success) / self.counts[strategy]
        self.mean_times[strategy] = ((self.mean_times[strategy] * (self.counts[strategy]-1)) + time) / self.counts[strategy]
        self.step_idx += 1

        truncated = self.step_idx >= self.instances_per_episode
        reward = success - time * self.curr_chi  # Reward for success, penalty for time taken

        obs = self._build_obs()

        info = {
            "strategy": strategy_name,
            "depth": depth if strategy_name=="change_path_dt" else None,
            "instance_id": instance_id,
            "with_xai": with_xai,
            "feature_changed": feat,
            "delta": delta,
            "original_instance": instance,
            "modified_instance": new_instance,
            "AI prediction (original)": actual_label,
            "AI prediction (cf)": new_pred,
            "success": success,
            "time": time,
        }

        return obs, reward, False, truncated, info


In [ ]:
import os
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv
import torch.nn as nn
import time


## Run training loop

In [ ]:
training_cog_params = {
    # Memory (used by _make_memory)
    "memory_recall_threshold": [-2.0, 0.5],
    "random_response_rate": [0.1, 0.5],
    "counterfactual_overshoot_fraction": [0.0, 0.5],

    # lr_calculation parameters
    # "read_seconds_per_item": 1.0,
    # "mental_calculation_seconds": 0.0,
    # "decision_boundary": [0.6, 1.7], #[0.8, 1.4],
    # "decision_noise": [0.7, 1.1], #[0.8, 1.2], #[0.5, 1.2],
    # "evidence_normalizer": "l2",     # fixed is fine
    # "displayed_significant_figures": 2.0,
    # "random_response_rate": 0.05, #[0.01, 0.2],

    "time_penalty_weight": [0.0, 0.05], #[0.0, 0.03],
}


In [ ]:
def train_counterfactual_rl_model(
    total_timesteps: int = 50_000,
    n_envs: int = 4,
    save_path: str = "./model_counterfactual/simple_chi_model.zip",
    log_path: str = None,
    ppo_kwargs: dict = None,
) -> PPO:
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    def make_env(rank: int):
        def _init():
            env = CounterfactualEnv(
                ai_dataset_loaders=ai_dataset_loaders,
                ais=ais,
                transforms=transforms,
                lr_exps=lr_exps,
                dt_exps=dt_exps,
                cog_params=training_cog_params,
                instances_per_episode=40,
                max_features=6
            )
            return Monitor(env)
        return _init

    train_env = SubprocVecEnv([make_env(i) for i in range(n_envs)])
    eval_env = SubprocVecEnv([make_env(999)])

    # train_env = DummyVecEnv([make_env(i) for i in range(n_envs)])
    # eval_env = DummyVecEnv([make_env(999)])

    eval_env.training = False
    
    default_kwargs = dict(
        policy="MlpPolicy",
        env=train_env,
        tensorboard_log=log_path,
        policy_kwargs=dict(
            net_arch=[dict(pi=[64, 64], vf=[64, 64])],
            activation_fn=nn.Tanh,
        ),
    )
    if ppo_kwargs:
        default_kwargs.update(ppo_kwargs)

    model = PPO(**{**default_kwargs, 'verbose': 0})
    
        # --------- Eval callback ---------
    eval_freq = max(int(10000 / max(n_envs, 1)), 1)
    eval_callback = EvalCallback(
        eval_env,
        eval_freq=eval_freq,
        best_model_save_path=os.path.dirname(save_path),
    )



    start_time = time.time()
    model.learn(total_timesteps=int(total_timesteps), callback=eval_callback, progress_bar=False)
    end_time = time.time()
    model.save(save_path)

    train_env.close()
    eval_env.close()

    return model


if __name__ == "__main__":
    model = train_counterfactual_rl_model(
        total_timesteps=4e5,
        n_envs=8,
        ppo_kwargs=dict(
            learning_rate=3e-4,
            ent_coef=0.01,
            gamma=0.8,
            device="cpu",  # set to 'cuda' if you have a GPU
            n_steps=1024
        ),
    )


## Evaluation plots

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter

def _sample_cog(env_cog_params, rng, fixed=None):
    fixed = fixed or {}
    sampled = {}
    for k, v in env_cog_params.items():
        if k == "time_penalty_weight":  # handled per-episode by env, but allow explicit pin
            if k in fixed:
                sampled[k] = float(fixed[k])
            else:
                # leave None → env will randomize time_penalty_weight in [chi_low, chi_high]
                continue
        elif k in fixed:
            sampled[k] = float(fixed[k])
        elif isinstance(v, (tuple, list)) and len(v) == 2:
            sampled[k] = float(rng.uniform(v[0], v[1]))
        elif isinstance(v, (int, float)):
            sampled[k] = float(v)
    return sampled

def evaluate_random_sweep(
    model,
    base_env_ctor,                 # callable returning a CounterfactualEnv; see factory below
    xai_types=('DT','LR','DT+LR'),
    episodes_per_type=12,
    with_xai_prob=0.5,
    fixed_cog_params=None,         # dict like {"memory_recall_threshold": -0.5, "time_penalty_weight": 0.01}
    seed=2025,
):
    rng = np.random.default_rng(seed)

    trial_rows = []
    episode_rows = []

    for xt in xai_types:
        for ep in range(episodes_per_type):
            # build env with overrides for this episode
            # we ask the factory for a fresh env to avoid state bleed
            env = base_env_ctor(eval_overrides=dict(
                xai_type=xt,
                with_xai_prob=with_xai_prob,
                cog_params_fixed=_sample_cog(base_env_ctor.cog_params, rng, fixed=fixed_cog_params),
            ))
            obs, _ = env.reset(seed=int(rng.integers(0, 2**31-1)))

            # Prepare tallies
            strat_counts = Counter()
            successes, times, rewards = [], [], []
            shown_types = Counter()
            with_xai_counts = Counter()

            while True:
                action, _ = model.predict(obs, deterministic=True)
                obs, reward, terminated, truncated, info = env.step(action)

                # record a trial
                trial_rows.append({
                    "episode": ep,
                    "xai_type": xt,
                    "with_xai": int(info["with_xai"]),
                    "xai_type_shown": env.xai_type_shown,
                    "strategy": info.get("strategy"),
                    "success": info.get("success", 0),
                    "time": info.get("time", 0.0),
                    "reward": reward,
                    # snapshot current cog params & time_penalty_weight for post-hoc grouping
                    **{f"cog_{k}": float(env.current_cog_params.get(k)) for k in env.varied_param_names},
                    "cog_chi": float(env.curr_chi),
                })

                # aggregate
                with_xai_counts[int(info["with_xai"])] += 1
                shown_types[str(env.xai_type_shown)] += 1
                if info.get("strategy") is not None:
                    strat_counts[info["strategy"]] += 1
                successes.append(info.get("success", 0))
                times.append(info.get("time", 0.0))
                rewards.append(reward)

                if truncated or terminated:
                    break

            steps = sum(strat_counts.values()) or 1
            row = {
                "xai_type": xt,
                "episode": ep,
                "steps": steps,
                "success_rate": float(np.mean(successes)) if successes else np.nan,
                "mean_time": float(np.mean(times)) if times else np.nan,
                "mean_reward": float(np.mean(rewards)) if rewards else np.nan,
                "with_xai_frac": with_xai_counts.get(1,0) / max(1,sum(with_xai_counts.values())),
                "shown_DT_frac": shown_types.get("DT",0) / max(1,sum(shown_types.values())),
                "shown_LR_frac": shown_types.get("LR",0) / max(1,sum(shown_types.values())),
                # carry representative cog params (last values from env)
                **{f"cog_{k}": float(env.current_cog_params.get(k)) for k in env.varied_param_names},
                "cog_chi": float(env.curr_chi),
            }
            # normalize strategy shares
            for sid, sname in strategies.items():
                row[f"use_{sname}"] = strat_counts.get(sname, 0) / steps
            episode_rows.append(row)

            env.close()

    df_trials = pd.DataFrame(trial_rows)
    df_episodes = pd.DataFrame(episode_rows)
    return df_trials, df_episodes


In [ ]:
def make_env_factory(
    ai_dataset_loaders, ais, transforms, lr_exps, dt_exps, cog_params,
    instances_per_episode=40, max_features=6
):
    def ctor(eval_overrides=None):
        return CounterfactualEnv(
            ai_dataset_loaders=ai_dataset_loaders,
            ais=ais,
            transforms=transforms,
            lr_exps=lr_exps,
            dt_exps=dt_exps,
            cog_params=cog_params,
            instances_per_episode=instances_per_episode,
            max_features=max_features,
            eval_overrides=eval_overrides or {}
        )
    # stash for sampling helper
    ctor.cog_params = cog_params
    return ctor


In [ ]:
base_env_ctor = make_env_factory(
    ai_dataset_loaders={1: ai_dataset_loaders[1]},  # only loan for now
    ais={1: ais[1]},
    transforms={1: transforms[1]},
    lr_exps={1: lr_exps[1]},
    dt_exps={1: dt_exps[1]},
    cog_params=training_cog_params,
    instances_per_episode=40,
    max_features=6
)

df_trials, df_episodes = evaluate_random_sweep(
    model=model,
    base_env_ctor=base_env_ctor,
    xai_types=(0,1,2),
    episodes_per_type=24,           # up it if you want tighter CIs
    with_xai_prob=0.5,
    fixed_cog_params=None,          # or e.g. {"memory_recall_threshold": -0.5, "time_penalty_weight": 0.01}
    seed=42
)


In [ ]:
import matplotlib.pyplot as plt

def plot_strategy_mix_by_xai(df_episodes):
    strat_cols = [c for c in df_episodes.columns if c.startswith("use_")]
    by_type = df_episodes.groupby("xai_type")[strat_cols].mean().sort_index()

    ax = by_type.plot(kind="bar", stacked=True, figsize=(9,4))
    ax.set_ylabel("Average share of steps")
    ax.set_xlabel("XAI type")
    ax.set_title("Strategy mix by XAI type (episode-averaged)")
    ax.legend(title="Strategy", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

plot_strategy_mix_by_xai(df_episodes)


In [ ]:
def plot_strategy_vs_param(df_episodes, param="cog_chi", q=6):
    types = sorted(df_episodes["xai_type"].unique())
    strat_cols = [c for c in df_episodes.columns if c.startswith("use_")]

    fig, axes = plt.subplots(1, len(types), figsize=(5*len(types),4), sharey=True)
    if len(types) == 1:
        axes = [axes]

    for ax, xt in zip(axes, types):
        sub = df_episodes[df_episodes["xai_type"] == xt].copy()
        # quantile bins
        sub["param_bin"] = pd.qcut(sub[param], q=q, duplicates="drop")
        grp = sub.groupby("param_bin")[strat_cols].mean()

        for col in strat_cols:
            ax.plot(range(len(grp)), grp[col].values, marker="o", label=col.replace("use_",""))

        ax.set_title(f"{xt} — {param}")
        ax.set_xlabel(f"{param} (quantile bins)")
        ax.set_xticks(range(len(grp)))
        ax.set_xticklabels([f"{i+1}" for i in range(len(grp))])
        ax.grid(True, alpha=0.3)

    axes[0].set_ylabel("Average share of steps")
    handles, labels = axes[-1].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=min(5, len(strat_cols)))
    fig.suptitle(f"Strategy usage vs {param} by XAI type")
    plt.tight_layout(rect=[0,0,1,0.92])
    plt.show()

# Examples:
plot_strategy_vs_param(df_episodes, param="cog_chi", q=6)
plot_strategy_vs_param(df_episodes, param="cog_memory_recall_threshold", q=6)
plot_strategy_vs_param(df_episodes, param="cog_random_response_rate", q=6)
plot_strategy_vs_param(df_episodes, param="cog_counterfactual_overshoot_fraction", q=6)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def plot_success_vs_param(df_episodes, param="cog_chi", q=6, metric="success_rate"):
    types = sorted(df_episodes["xai_type"].unique())
    fig, axes = plt.subplots(1, len(types), figsize=(5*len(types),4), sharey=True)
    if len(types) == 1:
        axes = [axes]
    for ax, xt in zip(axes, types):
        sub = df_episodes[df_episodes["xai_type"] == xt].copy()
        sub = sub[np.isfinite(sub[param])]
        if sub.empty:
            ax.set_title(f"{xt} — no data"); continue
        # quantile bins
        sub["param_bin"] = pd.qcut(sub[param], q=q, duplicates="drop")
        grp = sub.groupby("param_bin")[metric].mean()
        ax.plot(range(len(grp)), grp.values, marker="o")
        ax.set_title(f"{xt} — {param}")
        ax.set_xlabel(f"{param} (quantile bins)")
        ax.set_xticks(range(len(grp)))
        ax.set_xticklabels([f"{i+1}" for i in range(len(grp))])
        ax.grid(True, alpha=0.3)
    axes[0].set_ylabel(f"Mean {metric}")
    fig.suptitle(f"{metric} vs {param} by XAI type")
    plt.tight_layout(rect=[0,0,1,0.93])
    plt.show()

# examples:
plot_success_vs_param(df_episodes, param="cog_chi")
plot_success_vs_param(df_episodes, param="cog_memory_recall_threshold")
plot_success_vs_param(df_episodes, param="cog_counterfactual_overshoot_fraction")


## Load Participant data

In [ ]:
class ParticipantDataLoader:
    """
    A class to load and manage participant data from the wine_quality.csv file.
    Allows easy access to trials for specific participants and phases.
    """
    
    def __init__(self, csv_path: str):
        """
        Initialize the data loader with the CSV file.
        
        Args:
            csv_path (str): Path to the wine_quality.csv file
        """
        self.data = pd.read_csv(csv_path)
        self.participants = self._get_participant_info()
        
    def _get_participant_info(self) -> Dict:
        """Get unique participant information."""
        participant_info = {}
        for participant_id in self.data['Participant Id'].unique():
            participant_data = self.data[self.data['Participant Id'] == participant_id].iloc[0]
            participant_info[participant_id] = {
                'condition': participant_data['Condition'],
                'model': participant_data['Model'],
                'app_id': participant_data['AppId'],
                'complexity': participant_data['Complexity']
            }
        return participant_info
    
    def get_participant_ids(self) -> List:
        """Get list of all participant IDs."""
        return list(self.participants.keys())
    
    def get_participant_info(self, participant_id) -> Dict:
        """Get general info for a specific participant."""
        return self.participants.get(participant_id, {})
    
    def get_participant_trials(self, participant_id, phase: Optional[str] = None) -> pd.DataFrame:
        """
        Get all trials for a specific participant.
        
        Args:
            participant_id: The participant ID
            phase (str, optional): Filter by phase ('forward' or 'counterfactual')
            
        Returns:
            pd.DataFrame: Filtered trial data
        """
        participant_data = self.data[self.data['Participant Id'] == participant_id]
        
        if phase:
            participant_data = participant_data[participant_data['Phase'] == phase]
            
        return participant_data.sort_values('Trial Index')
    
    def get_forward_trials(self, participant_id) -> pd.DataFrame:
        """
        Get forward phase trials for a participant with relevant columns.
        
        Args:
            participant_id: The participant ID
            
        Returns:
            pd.DataFrame: Forward trial data with relevant columns
        """
        forward_data = self.get_participant_trials(participant_id, 'forward')
        
        forward_columns = [
            'Participant Id', 'Trial Index', 'Instance Id', 'XAIType', 'Tested w/ XAI', 'Time',
            'Response', 'AI prediction', 'DT prediction', 'LR prediction', 
            'Explainer prediction', 'Response==AI', 'Response==DT', 
            'Response==LR', 'Response==Explainer'
        ]
        
        # Only include columns that exist in the data
        available_columns = [col for col in forward_columns if col in forward_data.columns]
        
        return forward_data[available_columns]
    
    def get_counterfactual_trials(self, participant_id) -> pd.DataFrame:
        """
        Get counterfactual phase trials for a participant with relevant columns.
        
        Args:
            participant_id: The participant ID
            
        Returns:
            pd.DataFrame: Counterfactual trial data with relevant columns
        """
        cf_data = self.get_participant_trials(participant_id, 'counterfactual')
        
        cf_columns = [
            'Participant Id', 'Trial Index', 'Instance Id', 'XAIType', 'Tested w/ XAI', 'AI prediction', 'Time',
            'Changed feature index', 'Changed feature name', 'Changed feature type',
            'Changed from', 'Changed to', 'Changed amount', 'DT prediction (CF)',
            'LR prediction (CF)', 'Changed prediction (DT)', 'Changed prediction (LR)',
            'Changed explainer', 'Changed AI prediction', 'AI prediction (CF)',
        ]
        
        # Only include columns that exist in the data
        available_columns = [col for col in cf_columns if col in cf_data.columns]
        
        return cf_data[available_columns]
    
    def get_trial_by_index(self, participant_id, trial_index: int) -> pd.DataFrame:
        """
        Get a specific trial by trial index for a participant.
        
        Args:
            participant_id: The participant ID
            trial_index (int): The trial index number
            
        Returns:
            pd.DataFrame: Single trial data
        """
        participant_data = self.get_participant_trials(participant_id)
        return participant_data[participant_data['Trial Index'] == trial_index]
    
    def get_xai_performance(self, participant_id, xai_type: str = None) -> pd.DataFrame:
        """
        Get trials filtered by XAI type and whether XAI was shown.
        
        Args:
            participant_id: The participant ID
            xai_type (str, optional): Filter by specific XAI type
            
        Returns:
            pd.DataFrame: Filtered trial data
        """
        participant_data = self.get_participant_trials(participant_id)
        
        if xai_type:
            participant_data = participant_data[participant_data['XAIType'] == xai_type]
            
        return participant_data
    
    def summarize_participant_performance(self, participant_id) -> Dict:
        """
        Get a summary of participant performance across both phases.
        
        Args:
            participant_id: The participant ID
            
        Returns:
            Dict: Summary statistics
        """
        forward_trials = self.get_forward_trials(participant_id)
        cf_trials = self.get_counterfactual_trials(participant_id)
        
        summary = {
            'participant_info': self.get_participant_info(participant_id),
            'total_trials': len(self.get_participant_trials(participant_id)),
            'forward_trials': len(forward_trials),
            'counterfactual_trials': len(cf_trials),
            'avg_time_forward': forward_trials['Time'].mean() if not forward_trials.empty else 0,
            'avg_time_counterfactual': cf_trials['Time'].mean() if not cf_trials.empty else 0,
        }
        
        # Add accuracy metrics for forward trials if available
        if not forward_trials.empty:
            accuracy_columns = ['Response==AI', 'Response==DT', 'Response==LR', 'Response==Explainer']
            for col in accuracy_columns:
                if col in forward_trials.columns:
                    summary[f'accuracy_{col.split("==")[1].lower()}'] = forward_trials[col].mean()
        
        return summary
    
    def get_instance_data(self, instance_id: int) -> pd.DataFrame:
        """
        Get all trials for a specific instance across all participants.
        
        Args:
            instance_id (int): The instance ID
            
        Returns:
            pd.DataFrame: All trials for the instance
        """
        return self.data[self.data['Instance Id'] == instance_id]
    
    def compare_participants(self, participant_ids: List) -> pd.DataFrame:
        """
        Compare performance metrics across multiple participants.
        
        Args:
            participant_ids (List): List of participant IDs to compare
            
        Returns:
            pd.DataFrame: Comparison metrics
        """
        comparison_data = []
        
        for pid in participant_ids:
            summary = self.summarize_participant_performance(pid)
            comparison_data.append({
                'Participant Id': pid,
                **summary['participant_info'],
                'Total Trials': summary['total_trials'],
                'Forward Trials': summary['forward_trials'],
                'Counterfactual Trials': summary['counterfactual_trials'],
                'Avg Time Forward': summary['avg_time_forward'],
                'Avg Time Counterfactual': summary['avg_time_counterfactual'],
                **{k: v for k, v in summary.items() if k.startswith('accuracy_')}
            })
        
        return pd.DataFrame(comparison_data)
    

user_loader = ParticipantDataLoader(os.path.join(data_dir, 'combined.csv'))


## Fit to participant data

In [ ]:
# ---- GPBO with per-trial save of the best-NLL run ----
import numpy as np, pandas as pd, random, math, os
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C, WhiteKernel
from scipy.stats import norm

FEAT_COL, DELTA_COL = 'Changed feature index', 'Changed amount'
xai_idx = {v:k for k,v in XAI_types.items()}

def _get_p(fd): return fd.get('selected_p', fd.get('p_selected', 0.0))
def _nll_for_choice(out_probs, chosen_feat, random_response_rate, num_features=6):
    sm = smooth_probs_with_random_response(out_probs, random_response_rate=random_response_rate, num_features=num_features)
    p = max(min(float(_get_p(sm.get(chosen_feat, {}))), 1-1e-9), 1e-9)
    return -math.log(p)

def _ei(mu, sigma, best, xi=0.01):
    imp = best - mu - xi
    Z = np.divide(imp, sigma, out=np.zeros_like(mu), where=(sigma>0))
    return np.where(sigma>0, imp*norm.cdf(Z) + sigma*norm.pdf(Z), 0.0)

# Pass the EXACT training_cog_params you used to train PPO
def make_obs_builder(training_cog_params, strategies, XAI_types):
    # order of varied params in env: all (lo,hi) except 'time_penalty_weight', in dict order
    varied_param_names = []
    for k, v in training_cog_params.items():
        if k == "time_penalty_weight": 
            continue
        if isinstance(v, (list, tuple)) and len(v) == 2:
            varied_param_names.append(k)

    # map strings -> keys used by env (0:'DT', 1:'LR', 2:'DT+LR')
    xai_key_from_name = {v:k for k,v in XAI_types.items()}
    xai_key_from_name_shown = xai_key_from_name  # same map

    def build_obs(curr_chi, step_idx, with_xai, condition_name, shown_name,
                  counts, success_rates, mean_times, current_cog_params):
        # condition and shown are KEYS in env obs
        cond_key  = float(xai_key_from_name[condition_name])
        shown_key = float(xai_key_from_name_shown[shown_name])

        obs = [float(curr_chi), float(step_idx), float(with_xai), cond_key, shown_key]

        # triplets per strategy id order
        for sid in strategies.keys():  # assumes stable int keys 0..K-1
            obs += [
                float(counts.get(sid, 0)),
                float(success_rates.get(sid, 0.0)),
                float(mean_times.get(sid, 0.0)),
            ]

        # tail: varied params in EXACT training order
        for name in varied_param_names:
            obs.append(float(current_cog_params.get(name, 0.0)))

        return np.array(obs, dtype=np.float32), varied_param_names

    return build_obs, varied_param_names


def score_participant_with_theta(model, user_loader, participant_id, ai_dataset_loader,
                                 lr_df, dt_df, metadata_df, theta, random_response_rate=0.1):
    rt, counterfactual_overshoot_fraction, time_penalty_weight = theta['memory_recall_threshold'], theta['counterfactual_overshoot_fraction'], theta['time_penalty_weight']
    info = user_loader.get_participant_info(participant_id)
    app_id, model_name = info['app_id'], info['model']
    condition, complexity = info['condition'], info['complexity']
    ai_loader = filter_by_app_and_model(ai_dataset_loader, app_id, model_name)
    bounds = ai_loader.get_bounds_for_app(app_id)
    transform, ai = load_transform_and_ai(app_id)

    lr_exp = LogisticRegressionInterpreter(lr_df, metadata_df, app_id, "mlp",
                                           variant=("sparse" if complexity=="low" else "dense"))
    dt_exp = DecisionTreeInterpreter(dt_df, metadata_df, app_id, "mlp",
                                     depth=(3 if complexity=='high' else 2))

    mem_lr = _make_memory(memory_recall_threshold=rt)
    mem_dt = _make_memory(memory_recall_threshold=rt)

    fwd = user_loader.get_forward_trials(participant_id)
    cf  = user_loader.get_counterfactual_trials(participant_id)

    add_lr_heuristic_to_memory(lr_exp, mem_lr)
    add_dt_to_memory(mem_dt, dt_exp)

    # prime memory from forward (LR heuristic refresh if with-XAI)
    for _, tr in fwd.iterrows():
        insts, _ = ai_loader.load_instances([tr['Instance Id']], normalize=True)
        x = insts[0]
        _, t, inf = lr_heuristic(x, mem_lr, lr_exp, read_seconds_per_item=1.0, decision_boundary=1.0, decision_noise=0.8)
        if str(tr['Tested w/ XAI']).lower().startswith('w'):
            refresh_lr_heuristic_in_memory(mem_lr, lr_exp, inf, actual=tr['AI prediction'])
        mem_lr.tick(t)

    for _, tr in fwd.iterrows():
        insts, _ = ai_loader.load_instances([tr['Instance Id']], normalize=False)
        x = insts[0]
        explanation_access_mode = 'read' if tr['Tested w/ XAI']=='w/ XAI' else 'retrieve'
        dt_traverse(x, mem_dt, dt_exp, explanation_access_mode=explanation_access_mode)
        if explanation_access_mode=='read':
            refresh_dt_path_in_memory(mem_dt, dt_exp, x)
        mem_dt.tick(t)

    counts = {k:0 for k in strategies.keys()}
    succ   = {k:0.0 for k in strategies.keys()}
    mtime  = {k:0.0 for k in strategies.keys()}
    step   = 0
    cond_idx = xai_idx[condition]

    trials_log = []
    NLLs, MAEs, TIMES = [], [], []

    obs_builder, varied_param_names = make_obs_builder(training_cog_params, strategies, XAI_types)

    current_cog_params = training_cog_params.copy()
    current_cog_params = {
        'memory_recall_threshold': theta['memory_recall_threshold'],
        'random_response_rate': random_response_rate,
        'counterfactual_overshoot_fraction': theta['counterfactual_overshoot_fraction'],
    }
    


    for ti, tr in cf.iterrows():
        iid, ai_pred = tr['Instance Id'], tr['AI prediction']
        feat_chosen  = f'a{tr[FEAT_COL]}'; delta_chosen = float(tr[DELTA_COL])
        try:
            (x_raw,_), (x_norm,_) = ai_loader.load_instances([iid], normalize=False), ai_loader.load_instances([iid], normalize=True)
        except:
            continue
        x_raw, x_norm = x_raw[0], x_norm[0]

        with_xai = 1 if tr['Tested w/ XAI'] in (1,'w/ XAI','with XAI','With XAI') else 0
        shown = tr.get('XAIType', None) if condition=='DT+LR' else condition
        if shown is None: shown = random.choice(['DT','LR']) if condition=='DT+LR' else condition
        shown_idx = xai_idx['DT'] if shown=='DT' else xai_idx['LR']

        obs, _ = obs_builder(
            curr_chi=theta['time_penalty_weight'],
            step_idx=step,
            with_xai=with_xai,
            condition_name=condition,
            shown_name=shown,
            counts=counts, success_rates=succ, mean_times=mtime,
            current_cog_params=current_cog_params
        )
        action, _ = model.predict(obs, deterministic=True)
        strat_id, depth = int(action[0]), int(action[1]); strat = strategies[strat_id]
        explanation_access_mode = ('read' if with_xai else 'retrieve')

        depth = int(round(theta.get("counterfactual_tree_depth", depth)))
        depth = max(0, min(depth, 2))

        
        if condition=='DT':
            if strat in ('zero_out_lr_displayed', 'zero_out_lr_heuristic', 'recall_change_lr'):
                strat = 'change_path_dt'  # fallback
        elif condition=='LR':
            if strat in ('change_path_dt', 'recall_change_dt'):
                strat = 'zero_out_lr_heuristic'  # fallback
        elif shown=='DT':
            if strat in ('zero_out_lr_displayed'):
                strat = 'change_path_dt'  # fallback

        if strat=='recall_change_dt' and with_xai==1:
            strat = 'change_path_dt' if np.random.random() < 0.9 else strat  # can't recall if you saw it


        # get feature probs + mean_delta (+time)
        if strat == "change_path_dt":
            out = cf_change_path_dt(x_raw, dt_exp, bounds, memory=mem_dt, counterfactual_tree_depth=depth, explanation_access_mode=explanation_access_mode, depth_choice_temperature=1.0)
            t = out.pop('expected_time', 0.0); mem_dt.tick(t)
            if int(dt_exp.apply_to_instance(x_raw)['class_index']) != ai_pred:
                for k in out: out[k]['mean_delta'] *= -1
        elif strat == "zero_out_lr_heuristic":
            out = cf_lr_heuristic(x_norm, mem_lr, lr_exp, bounds, retrieval_candidate_count=6, actual_label=ai_pred)
            t = out.pop('expected_time', 0.0); mem_lr.tick(t)
        elif strat == "zero_out_lr_displayed":
            out = cf_lr_calculation(x_raw, lr_exp, bounds=bounds, memory=mem_lr)
            t = out.pop('expected_time', 0.0); mem_lr.tick(t)
            if int(lr_exp.apply_to_instance(x_raw) > 0) != ai_pred:
                for k in out: out[k]['mean_delta'] *= -1
        elif strat == "recall_change_dt":
            out = recall_change_dt(x_raw, mem_dt, bounds=bounds, retrieved_combo_count=3)
            t = out.pop('expected_time', 0.0); mem_dt.tick(t)
        else:
            direction = 'increase' if ai_pred==0 else 'decrease'
            out = recall_change_lr(mem_lr, retrieved_combo_count=6, preferred_change_direction=direction)
            t = out.pop('expected_time', 0.0); mem_lr.tick(t)

        out = smooth_probs_with_random_response(out, random_response_rate=random_response_rate, num_features=6)

        # model's edit for participant-chosen feature (for MAE)
        try:
            int(float(feat_chosen[1:]))
        except:
            continue
        f_idx = int(float(feat_chosen[1:])) if isinstance(feat_chosen,str) and feat_chosen.startswith('a') else int(feat_chosen)
        feat_chosen = f'a{f_idx}'
        if f_idx in transform.categorical_feature_indices:
            delta_chosen = 1.0 if delta_chosen > 0 else -1.0
        edited = apply_change_to_feature(x_raw, feat_chosen, bounds, out[feat_chosen]['mean_delta'],
                                         counterfactual_overshoot_fraction=counterfactual_overshoot_fraction)
        delta_pred = edited[f_idx] - x_raw[f_idx]

        # NLL & MAE
        nll = _nll_for_choice(out, feat_chosen, random_response_rate=random_response_rate, num_features=6)
        # if nll>3:
        mae = abs(delta_pred - delta_chosen)
        mae /= (bounds[f'a{f_idx}'][1] - bounds[f'a{f_idx}'][0])  # normalize by range

        # sample model’s own feature/delta and compute flips
        feat_samp, delta_samp = sample_from_probs(out)   # returns (feat, delta)
        x_samp = apply_change_to_feature(x_raw, feat_samp, bounds, delta_samp, counterfactual_overshoot_fraction=counterfactual_overshoot_fraction)
        ai_cf  = run_ai_prediction(x_samp, transform, ai)
        if shown=='DT':
            xai_cf = int(dt_exp.apply_to_instance(x_samp)['class_index'])
        else:
            xai_cf = int(lr_exp.apply_to_instance(x_samp) > 0)

        counts[strat_id]+=1
        mtime[strat_id] = (mtime[strat_id]*(counts[strat_id]-1)+t)/counts[strat_id]
        step += 1

        NLLs.append(nll); MAEs.append(mae); TIMES.append(t)

        # ----- per-trial log row -----

        row = {
            **{k: (v.item() if hasattr(v, "item") else v) for k, v in tr.to_dict().items()},
            **{k: (v.item() if hasattr(v, "item") else v) for k, v in info.items()},
            "Participant Id": participant_id,
            "Trial Index": int(ti),

            # model-sampled edit summary
            "Model strategy": strat,
            "Model depth": int(depth) if strat == "change_path_dt" else None,
            "Model changed feature index": feat_samp[1:] if isinstance(feat_samp,str) and feat_samp.startswith('a') else int(feat_samp),
            "Model changed feature name": str(feat_samp),
            "Model changed amount": float(delta_samp),
            "Model AI prediction (CF)": int(ai_cf),
            "Model changed AI prediction": int(ai_cf != ai_pred),
            "Model XAI prediction (CF)": int(xai_cf),
            "Model changed XAI prediction": int(xai_cf != ai_pred),

            # per-participant-feature comparison
            "Model mean_delta for chosen feature": float(out.get(feat_chosen, {}).get("mean_delta", 0.0)),
            "Trial NLL": float(nll),
            "Trial MAE": float(mae),
            "Trial time": float(t),                # <-- add this line

            # hyperparams
            "memory_recall_threshold": float(rt),
            "counterfactual_overshoot_fraction": float(counterfactual_overshoot_fraction),
            "time_penalty_weight": float(time_penalty_weight),
        }

        trials_log.append(row)
    return dict(
        nll=float(np.mean(NLLs)) if NLLs else 1e3,
        mae=float(np.mean(MAEs)) if MAEs else 1e3,
        time=float(np.mean(TIMES)) if TIMES else 0.0,
        trials=trials_log
    )

def fit_participant_with_gpbo(model, user_loader, participant_id, ai_dataset_loader,
                              lr_df, dt_df, metadata_df, n_init=8, n_iter=30, random_response_rate=0.1,
                              bounds=dict(memory_recall_threshold=(-2.0,0.5), counterfactual_overshoot_fraction=(0.0,0.5), time_penalty_weight=(0.0,0.02)),
                              alpha_mae=1.0, beta_time=0.0, seed=0):
    rng = np.random.default_rng(seed)
    def obj(theta):
        lap = theta.get('random_response_rate', random_response_rate)
        s = score_participant_with_theta(model, user_loader, participant_id, ai_dataset_loader,
                                         lr_df, dt_df, metadata_df, theta, random_response_rate=lap)
        score = s['nll'] + alpha_mae*s['mae'] + beta_time*(theta['time_penalty_weight']*s['time'])
        return score, s

    X, y = [], []
    best = {'obj': np.inf}; best_trials = []
    def sample_theta():
        return dict(
            memory_recall_threshold=rng.uniform(*bounds['memory_recall_threshold']),
            counterfactual_overshoot_fraction=rng.uniform(*bounds['counterfactual_overshoot_fraction']),
            time_penalty_weight=rng.uniform(*bounds['time_penalty_weight']),
            random_response_rate=rng.uniform(*bounds['random_response_rate']),
            depth=rng.uniform(*bounds.get('depth',(0.0,2.0)))
        )

    # init
    for _ in range(n_init):
        th = sample_theta(); val, s = obj(th)
        X.append([th['memory_recall_threshold'], th['counterfactual_overshoot_fraction'], th['time_penalty_weight'], th['random_response_rate'], th['depth']]); y.append(val)
        if val < best['obj']:
            best = {**th, 'obj': float(val), 'nll': s['nll'], 'mae': s['mae'], 'time': s['time']}
            best_trials = s['trials']
    X, y = np.array(X), np.array(y)

    gp = GaussianProcessRegressor(
        kernel=C(1.0,(1e-3,1e3)) * Matern(length_scale=[0.5,0.2,0.01,0.05, 0.8], nu=2.5) + WhiteKernel(1e-4,(1e-8,1e-1)),
        alpha=1e-6, normalize_y=True, n_restarts_optimizer=3, random_state=seed
    ).fit(X,y)

    for _ in range(n_iter):
        grid = np.stack([
            rng.uniform(*bounds['memory_recall_threshold'], size=200),
            rng.uniform(*bounds['counterfactual_overshoot_fraction'], size=200),
            rng.uniform(*bounds['time_penalty_weight'], size=200),
            rng.uniform(*bounds['random_response_rate'], size=200),
            rng.uniform(*bounds.get('depth',(0.0,2.0)), size=200),
        ], axis=1)
        mu, sig = gp.predict(grid, return_std=True)
        x_next = grid[np.argmax(_ei(mu, sig, best=np.min(y), xi=0.01))]
        th = dict(memory_recall_threshold=x_next[0], counterfactual_overshoot_fraction=x_next[1], time_penalty_weight=x_next[2], random_response_rate=x_next[3], depth=int(round(x_next[4])))
        val, s = obj(th)
        X = np.vstack([X, x_next]); y = np.append(y, val); gp.fit(X,y)
        if val < best['obj']:
            best = {**th, 'obj': float(val), 'nll': s['nll'], 'mae': s['mae'], 'time': s['time']}
            best_trials = s['trials']

    # pack best + trials so you don't need to re-run
    return {**best, 'participant_id': participant_id, 'trials': best_trials, 'X': X, 'y': y}

# ---- Batch: many participants, save one CSV of all best-trial logs ----
def fit_many_and_save_csv(model, user_loader, participant_ids, ai_dataset_loader,
                          lr_df, dt_df, metadata_df, out_csv="rl_fit_trials.csv",
                          **gpbo_kwargs):
    all_rows, summaries = [], []
    i = 0
    for pid in participant_ids:
        i += 1
        res = fit_participant_with_gpbo(model, user_loader, pid, ai_dataset_loader,
                                        lr_df, dt_df, metadata_df, **gpbo_kwargs)
        # append trials with summary fields
        for r in res['trials']:
            all_rows.append({
                **r,
                'Best NLL': res['nll'],
                'Best MAE': res['mae'],
                'Best time': res['time'],
                'Best memory_recall_threshold': res['memory_recall_threshold'],
                'Best counterfactual_overshoot_fraction': res['counterfactual_overshoot_fraction'],
                'Best time_penalty_weight': res['time_penalty_weight'],
                'Participant Id': res['participant_id'],
            })
        summaries.append({k: res[k] for k in ('participant_id','nll','mae','time','memory_recall_threshold','counterfactual_overshoot_fraction','time_penalty_weight','random_response_rate','depth','obj')})

    df_trials = pd.DataFrame(all_rows)
    os.makedirs(os.path.dirname(out_csv) or ".", exist_ok=True)
    df_trials.to_csv(out_csv, index=False)
    return df_trials, pd.DataFrame(summaries)

# -------- Example usage --------
participant_list = [p for p in user_loader.get_participant_ids()
                    if user_loader.get_participant_info(p)['condition'] in ('LR','DT','DT+LR')]
subset = random.sample(participant_list, k=min(50, len(participant_list)))

model = PPO.load("model_counterfactual/simple_chi_model.zip")
df_trials, df_summary = fit_many_and_save_csv(
    model=model,
    user_loader=user_loader,
    participant_ids=subset,
    ai_dataset_loader=ai_dataset_loader,
    lr_df=lr_df, dt_df=dt_df, metadata_df=metadata_df,
    out_csv="outputs/rl_fit_trials.csv",
    n_init=8, n_iter=24, random_response_rate=0.1,
    bounds=dict(memory_recall_threshold=(-2.0,0.5), counterfactual_overshoot_fraction=(0.05,0.5), time_penalty_weight=(0.0,0.02), random_response_rate=(0, 1.0), depth=(0.0, 2.0)),
    alpha_mae=2.0, beta_time=0.0, seed=123
)


In [ ]:
# remove subset from participant ids
participant_list = [p for p in participant_list if p not in subset]
subset = random.sample(participant_list, k=min(120, len(participant_list)))


df_trials, df_summary = fit_many_and_save_csv(
    model=model,
    user_loader=user_loader,
    participant_ids=subset,
    ai_dataset_loader=ai_dataset_loader,
    lr_df=lr_df, dt_df=dt_df, metadata_df=metadata_df,
    out_csv="outputs/rl_fit_trials_round2.csv",
    n_init=8, n_iter=24, random_response_rate=0.1,
    bounds=dict(memory_recall_threshold=(-2.0,0.5), counterfactual_overshoot_fraction=(0.05,0.5), time_penalty_weight=(0.0,0.02), random_response_rate=(0, 1.0), depth=(0.0, 2.0)),
    alpha_mae=2.0, beta_time=0.0, seed=123
)


In [ ]:
current_results = pd.read_csv("outputs/current_participants.csv")
current_participants = set(current_results['Participant Id'].unique())
new_participants = [p for p in participant_list if p not in current_participants]


In [ ]:
# participant_list = [p for p in participant_list if p not in subset]
# subset = random.sample(participant_list, k=min(120, len(participant_list)))


df_trials, df_summary = fit_many_and_save_csv(
    model=model,
    user_loader=user_loader,
    participant_ids=new_participants,
    ai_dataset_loader=ai_dataset_loader,
    lr_df=lr_df, dt_df=dt_df, metadata_df=metadata_df,
    out_csv="outputs/rl_fit_trials_round3.csv",
    n_init=8, n_iter=24, random_response_rate=0.1,
    bounds=dict(memory_recall_threshold=(-2.0,0.5), counterfactual_overshoot_fraction=(0.05,0.5), time_penalty_weight=(0.0,0.02), random_response_rate=(0, 1.0), depth=(0.0, 2.0)),
    alpha_mae=2.0, beta_time=0.0, seed=123
)
